# 🚀 Surget.AI — Free Google Colab GPU VLM Server
### Zero-Cost Tier-2 Vision-Language Grounding Server for Autonomous Agents
**Hardware:** Free Google Colab T4 GPU (15GB VRAM) or A100 GPU

This notebook serves **Qwen2-VL** or **UI-TARS** over a public HTTPS tunnel.
Your local Windows machine runs **Tier-0 OS Grounding** (1.7µs) and **Workflow Memory** locally, and routes Tier-2 visual queries to this Colab GPU!

---

## ⚙️ Step 1: Check GPU & Install Dependencies
Ensure your Colab runtime is set to **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi
!pip install -q transformers accelerate torchvision pillow fastapi uvicorn pydantic nest_asyncio pyngrok

## 🧠 Step 2: Load Model (Qwen2-VL-2B / 7B or UI-TARS)
We use `Qwen/Qwen2-VL-2B-Instruct` by default (downloads in ~45 seconds, fast 35+ tokens/sec on free T4 GPU) or you can switch to `Qwen/Qwen2-VL-7B-Instruct`.

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

# Choose model: 'Qwen/Qwen2-VL-2B-Instruct' (Fast & Free) or 'Qwen/Qwen2-VL-7B-Instruct'
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print(f"[+] Loading {MODEL_ID} into Colab GPU VRAM...")

processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=1280*28*28)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("[OK] Model successfully loaded on GPU!")

## 🌐 Step 3: Define FastAPI Grounding Server

In [ ]:
import io
import base64
import re
from PIL import Image
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import nest_asyncio
nest_asyncio.apply()

app = FastAPI(title="SurgetAI Colab VLM Server")

class GroundingRequest(BaseModel):
    instruction: str
    image_base64: str

@app.get("/")
@app.get("/health")
def health():
    return {"status": "ONLINE", "engine": "SurgetAI-Colab-VLM", "model": MODEL_ID, "cuda": torch.cuda.is_available()}

@app.post("/v1/ground")
def ground_ui_element(req: GroundingRequest):
    try:
        image_bytes = base64.b64decode(req.image_base64)
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        width, height = image.size
        
        prompt = (
            f"Look at this computer screen. Locate the exact UI element: '{req.instruction}'. "
            f"Provide the center coordinates of this element in format: [x, y] between [0, 1000]."
        )
        
        messages = [
            {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=64)
        
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]
        response_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        
        # Extract coordinates [x, y]
        match = re.search(r'\[\s*(\d+)\s*,\s*(\d+)\s*\]', response_text)
        if match:
            norm_x, norm_y = int(match.group(1)), int(match.group(2))
        else:
            norm_x, norm_y = 500, 500  # fallback center
            
        return {
            "status": "success",
            "instruction": req.instruction,
            "normalized_coords": [norm_x, norm_y],
            "pixel_coords": [int(norm_x / 1000 * width), int(norm_y / 1000 * height)],
            "raw_response": response_text
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("[OK] FastAPI Grounding Server initialized!")

## 🚀 Step 4: Expose Server with Cloudflare (Zero Signup Needed!)
Cloudflare gives an instant, free public HTTPS tunnel without needing any token or registration.

In [ ]:
import subprocess
import time
import threading
import uvicorn

# 1. Download cloudflared if not present
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 2. Launch Uvicorn in background thread
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

t = threading.Thread(target=start_server, daemon=True)
t.start()
time.sleep(2)

# 3. Launch Cloudflare Tunnel and print public URL
print("\n" + "="*65)
print("🌐 STARTING CLOUDFLARE PUBLIC TUNNEL...")
print("="*65)
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000